<a href="https://colab.research.google.com/github/kousiknandy/pycolab/blob/main/parse_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import csv

class CSVReducer:
    def __init__(self, file):
        self.file = file

    def reduce(self, column, op):
        with open(self.file, mode="r", encoding="utf-8") as f:
            c = csv.DictReader(f)
            res = 0
            for row in c:
                res = op(res, float(row[column]))
        return res

    def fastreduce(self, column, op):
        with open(self.file, mode="r", encoding="utf-8") as f:
            c = csv.reader(f)
            hdr = next(c)
            idx = hdr.index(column)
            res = 0
            for row in c:
                res = op(res, float(row[idx]))
        return  res

In [6]:
import operator

c = CSVReducer("/content/sample_data/california_housing_test.csv")
r = c.reduce("population", operator.add)
r


4208396.0

In [7]:
r = c.fastreduce("population", operator.add)
r

4208396.0

In [9]:
from multiprocessing import Pool
import os

def chunk_reader(file,  start, end):
    with open(file) as f:
        if start:
            f.seek(start-1)
            f.readline()
        while f.tell() < end:
            yield f.readline()

def process_chunk(file, col, start, end):
    f = chunk_reader(file,  start, end)
    res = 0
    c = csv.reader(f)
    if start == 0: next(c)
    for r in c:
        res += float(r[col])
    return res

def divide_file(file, pieces):
    file_size = os.path.getsize(file)
    chunksz = file_size // pieces
    offsets = [(i * chunksz, (i+1)*chunksz) for i in range(pieces)]
    offsets[-1] = (offsets[-1][0], file_size)
    return offsets

with Pool(3) as pool:
    res = pool.starmap(process_chunk, [("/content/sample_data/california_housing_test.csv",5, s, e) for s,e in divide_file("/content/sample_data/california_housing_test.csv",3)])

r = sum(res)
r

4208396.0